In [ ]:
def _get_chargebee_auth() -> tuple[str, str]:
    """Return (site, api_key). Raises if not configured."""
    site = (os.environ.get("CHARGEBEE_SITE") or "").strip()
    key = (os.environ.get("CHARGEBEE_API_KEY") or "").strip()
    if not site or not key:
        raise RuntimeError(
            "CHARGEBEE_SITE and CHARGEBEE_API_KEY must be set (or mount API key from Secret Manager)."
        )
    return site, key

In [ ]:
def resolve_invoice_status(invoice_ids: list[str]) -> dict[str, str]:
    """
    Return current status per invoice_id via Chargebee GET /invoices/{id}.
    Maps Chargebee status to 'unpaid'|'open'|'paid'|'cancelled'. Only 'unpaid' and 'open' are retried.
    """
    if not invoice_ids:
        return {}
    site, _ = _get_chargebee_auth()
    base_url = f"https://{site}.chargebee.com/api/v2/invoices"
    result: dict[str, str] = {}
    for i, inv_id in enumerate(invoice_ids):
        inv_id = str(inv_id).strip()
        if not inv_id:
            continue
        url = f"{base_url}/{urllib.parse.quote(inv_id, safe='')}"
        code, text = _chargebee_fetch(url, method="GET")
        if code == 200:
            try:
                data = json.loads(text)
                cb_status = (data.get("invoice") or {}).get("status", "")
                result[inv_id] = _CB_STATUS_TO_CANONICAL.get(
                    (cb_status or "").lower().replace("-", "_"), "unknown"
                )
            except (json.JSONDecodeError, TypeError):
                result[inv_id] = "unknown"
        elif code == 404:
            result[inv_id] = "cancelled"
        else:
            result[inv_id] = "unknown"
        if i < len(invoice_ids) - 1:
            time.sleep(0.2)
    return result